# Deliverable 3 — Complete Workflow: from a Folder of Images to Captions

This notebook runs the 3 trained models one after the other on **any folder of images** (no training):

```txt
 folder ──> 1. CLASSIFICATION (D1) ──> not a photo: set aside with its class
                     │ photo
                     ▼
            2. DENOISING (D2)
                     │
                     ▼
            3. CAPTIONING (D3, InceptionV3 + GRU) ──> results/pipeline_<folder>.csv + gallery
```

| Step | Model file | Input |
|---|---|---|
| 1. photo or not | `results/Hybrid_MobileNetV2.keras` | 224 x 224, pixels 0-255 |
| 2. denoising | `results/denoise_UNet_deep.keras` | 256 x 256, pixels 0-1 |
| 3a. image encoder | InceptionV3 ImageNet, layer `avg_pool` | 299 x 299, `preprocess_input` |
| 3b. caption | `results/caption_Hvass_GRU_InceptionV3.keras` + `..._vocab.json` | CNN vector + previous words |

**Demo day**: run everything, then change `INPUT_DIR` (and `ANNOTATIONS` if captions are given) in the last
section and run it again.

In [ ]:
import tensorflow as tf, keras
print("TensorFlow", tf.__version__, "| Keras", keras.__version__)

## 1. Configuration

In [ ]:
print("[Block 1] Configuration : starting...")

import json, os, re, time, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import imageio.v3 as iio
import tensorflow as tf
import keras
from keras import layers, ops
from keras.applications import ConvNeXtBase, InceptionV3, VGG16, convnext, inception_v3, vgg16
from nltk.translate.bleu_score import corpus_bleu

warnings.filterwarnings("ignore", module="nltk")

ROOT = Path("")
RAW = ROOT / "data_raw"
DATA = ROOT / "data"
RESULTS = ROOT / "results"

INPUT_DIR = RAW / "Dataset"   # folder to process
ANNOTATIONS = None            # optional: COCO .json or .csv (file_name, caption)

DENOISE = True                # False = skip the denoising step
DECODING = "beam"             # "beam" or "greedy"
BEAM = 3
BATCH = 32
# Captioning model trained in 03_captioning.ipynb (take the best one of its section 7):
#   "Hvass_GRU_InceptionV3" (v1), "Attention_GRU_InceptionV3" (v3),
#   "ShowAttendTell_InceptionV3" or "ShowAttendTell_ConvNeXtBase" (v4)
# Attention maps are shown only for the models with attention (v3, v4)
CAPTION_MODEL = "Attention_GRU_InceptionV3"
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".gif"}

# Same classes and order as deliverable 1
CLASS_NAMES = ["Painting", "Photo", "Schematics", "Sketch", "Text"]
PHOTO = CLASS_NAMES.index("Photo")

print("[Block 1] OK")

## 2. Loading the Models

The models are read from `results/`. If a file is missing, run the notebook that trains it.

In [ ]:
print("[Block 2] Loading the models : starting...")

# Custom layer of the v4 captioning models: same code as 03_captioning.ipynb (section 5), needed to load them
@keras.saving.register_keras_serializable(package="Leyenda")
class AttentionGRU(layers.Layer):
    """GRU that looks at the image regions before writing each word (Show, Attend and Tell).

    Inputs: [words (batch, T, E), regions (batch, R, D), initial state (batch, units)]
    Outputs: [GRU states (batch, T, units), contexts (batch, T, D), attention weights (batch, T, R)]
    """
    def __init__(self, units, **kwargs):
        super().__init__(**kwargs)
        self.units = units
        self.supports_masking = True   # padding steps are skipped (state unchanged)
        self.att_regions = layers.Dense(units, use_bias=False, name="att_regions")   # region -> attention space
        self.att_state = layers.Dense(units, name="att_state")                       # GRU memory -> attention space
        self.att_score = layers.Dense(1, use_bias=False, name="att_score")           # one score per region
        self.gate = layers.Dense(1, activation="sigmoid", name="gate")               # how much image to use (0 to 1)
        self.cell = layers.GRUCell(units, name="cell")                               # one GRU step

    def build(self, input_shape):
        words, regions, _ = input_shape
        self.att_regions.build(regions)
        self.att_state.build((None, self.units))
        self.att_score.build((None, None, self.units))
        self.gate.build((None, self.units))
        self.cell.build((None, words[-1] + regions[-1]))   # input of the GRU = word + context

    def call(self, inputs, mask=None, training=False):
        words, regions, h0 = inputs
        keys = self.att_regions(regions)   # computed once per image
        m = mask[0] if isinstance(mask, (list, tuple)) else mask
        m = ops.ones(ops.shape(words)[:2]) if m is None else ops.cast(m, words.dtype)
        u = self.units
        f = ops.shape(regions)[-1]

        def step(previous, x):
            # previous = [h, context, alpha] side by side (tf.scan needs one tensor)
            h, (word, real) = previous[:, :u], x
            # 1. Attention: a score for each region, from the GRU memory h (what was written so far)
            scores = self.att_score(ops.tanh(keys + self.att_state(h)[:, None, :]))[..., 0]
            alpha = ops.softmax(scores, axis=-1)                                # R weights that sum to 1
            context = self.gate(h) * ops.sum(alpha[..., None] * regions, axis=1)   # weighted mean of the regions
            # 2. GRU step: reads the word AND the context
            new_h = self.cell(ops.concatenate([word, context], axis=-1), [h], training=training)[1][0]
            h = real[:, None] * new_h + (1 - real[:, None]) * h               # padding: keep the old state
            return ops.concatenate([h, context, alpha], axis=-1)

        start = ops.concatenate([h0, ops.zeros_like(regions[:, 0]), ops.zeros_like(regions[:, :, 0])], axis=-1)
        out = tf.scan(step, (ops.transpose(words, (1, 0, 2)), ops.transpose(m, (1, 0))), initializer=start)
        out = ops.transpose(out, (1, 0, 2))   # (batch, T, units + D + R)
        return out[..., :u], out[..., u:u + f], out[..., u + f:]

    def compute_mask(self, inputs, mask=None):
        m = mask[0] if isinstance(mask, (list, tuple)) else mask
        return [m, m, m]

    def compute_output_shape(self, input_shape):
        words, regions, _ = input_shape
        return ((words[0], words[1], self.units), (words[0], words[1], regions[-1]),
                (words[0], words[1], regions[1]))

    def get_config(self):
        return {**super().get_config(), "units": self.units}

start = time.time()
classifier = keras.models.load_model(RESULTS / "Hybrid_MobileNetV2.keras", compile=False)
denoiser = keras.models.load_model(RESULTS / "denoise_UNet_deep.keras", compile=False)
with open(RESULTS / f"caption_{CAPTION_MODEL}_vocab.json") as f:
    caption_config = json.load(f)   # vocabulary + name of the CNN used in training
decoder = keras.models.load_model(RESULTS / caption_config["model"], compile=False)

def build_encoder(name):
    # Same CNN as in 03_captioning.ipynb (section 3)
    if name == "InceptionV3":
        cnn = InceptionV3(include_top=True, weights="imagenet")
        return keras.Model(cnn.input, cnn.get_layer("avg_pool").output, name="InceptionV3_avg_pool"), inception_v3.preprocess_input
    if name == "InceptionV3_grid":
        # 8 x 8 regions -> 64 vectors of 2048 numbers (attention model)
        cnn = InceptionV3(include_top=True, weights="imagenet")
        block = cnn.get_layer("mixed10").output
        grid = keras.layers.Reshape((-1, block.shape[-1]), name="regions")(block)
        return keras.Model(cnn.input, grid, name="InceptionV3_mixed10"), inception_v3.preprocess_input
    if name == "ConvNeXtBase_grid":
        # Last block before the classifier: 7 x 7 regions -> 49 vectors of 1024 numbers
        cnn = ConvNeXtBase(include_top=True, weights="imagenet")
        block = cnn.get_layer("convnext_base_head_gap").input                  # (7, 7, 1024)
        block = cnn.get_layer("convnext_base_head_layernorm")(block)           # same normalisation as the classifier
        grid = keras.layers.Reshape((-1, block.shape[-1]), name="regions")(block)
        return keras.Model(cnn.input, grid, name="ConvNeXtBase_grid"), convnext.preprocess_input
    if name == "VGG16":
        cnn = VGG16(include_top=True, weights="imagenet")
        return keras.Model(cnn.input, cnn.get_layer("fc2").output, name="VGG16_fc2"), vgg16.preprocess_input
    raise ValueError(f"unknown encoder {name}")

encoder, preprocess_input = build_encoder(caption_config["encoder"])

VOCAB = caption_config["vocab"]
MAX_WORDS = caption_config["max_words"]
TOKEN_START = VOCAB.index(caption_config["mark_start"])
TOKEN_END = VOCAB.index(caption_config["mark_end"])
CLF_SIZE = tuple(classifier.input_shape[1:3])     # 224 x 224
DENOISE_SIZE = tuple(denoiser.input_shape[1:3])   # 256 x 256
ENC_SIZE = tuple(encoder.input_shape[1:3])        # 299 x 299 (InceptionV3)
assert decoder.output_shape[-1] == len(VOCAB), "the vocabulary file does not match the captioning model"

for name, m in (("classifier", classifier), ("denoiser", denoiser), ("encoder", encoder), ("decoder", decoder)):
    print(f"  {name:10s} {m.name:22s} {m.count_params():>12,} weights | input {m.input_shape}")
print(f"  vocabulary: {len(VOCAB)} words, captions of at most {MAX_WORDS} words | loaded in {time.time() - start:.0f} s")

print("[Block 2] OK")

## 3. The Workflow Steps

Each function uses the same preprocessing as in the notebook where the model was trained.

In [ ]:
print("[Block 3] Workflow steps : definition...")

AUTOTUNE = tf.data.AUTOTUNE

def read_rgb(path):
    img = tf.io.decode_image(tf.io.read_file(path), channels=3, expand_animations=False)
    img.set_shape([None, None, 3])
    return tf.cast(img, tf.float32)

def classify(paths):
    # Step 1: probabilities of the 5 classes
    ds = tf.data.Dataset.from_tensor_slices(paths).map(lambda p: tf.image.resize(read_rgb(p), CLF_SIZE), AUTOTUNE)
    return classifier.predict(ds.batch(BATCH).prefetch(AUTOTUNE), verbose=0)

def load_01(paths):
    # Photos in 256 x 256, pixels in [0, 1]
    ds = tf.data.Dataset.from_tensor_slices(paths).map(
        lambda p: tf.clip_by_value(tf.round(tf.image.resize(read_rgb(p), DENOISE_SIZE, antialias=True)), 0, 255) / 255.0,
        AUTOTUNE)   # same as deliverable 2
    return np.concatenate([b.numpy() for b in ds.batch(BATCH)])

def denoise(images):
    # Step 2: denoising
    return np.concatenate([denoiser(images[s:s + BATCH], training=False).numpy() for s in range(0, len(images), BATCH)])

def encode(images):
    # Step 3a: CNN vectors
    x = tf.image.resize(np.asarray(images, dtype="float32") * 255.0, ENC_SIZE)
    return encoder.predict(preprocess_input(x), batch_size=BATCH, verbose=0)

# Step 3b: captions (same code as 03_captioning.ipynb, section 5.1)
def tokens_to_text(tokens):
    words = []
    for t in tokens[1:]:
        if t == TOKEN_END or t == 0:
            break
        words.append(VOCAB[t])
    return " ".join(words)

def next_word_probs(feats, tokens):
    out = decoder({"transfer_values_input": feats, "decoder_input": tokens}, training=False)
    return np.array(out[:, -1], dtype="float64")

def greedy_captions(features, max_words=MAX_WORDS):
    feats = np.asarray(features, dtype="float32")
    tokens = np.zeros((len(feats), max_words + 2), dtype="int32"); tokens[:, 0] = TOKEN_START
    done = np.zeros(len(feats), dtype=bool)
    for t in range(max_words + 1):
        probs = next_word_probs(feats, tokens[:, :t + 1]); probs[:, :2] = 0
        word = probs.argmax(axis=1); word[done] = 0
        tokens[:, t + 1] = word
        done |= word == TOKEN_END
        if done.all():
            break
    return [tokens_to_text(row) for row in tokens]

def beam_captions(features, beam=BEAM, max_words=MAX_WORDS, alpha=0.7):
    V = len(VOCAB)
    feats = np.repeat(np.asarray(features, dtype="float32"), beam, axis=0)
    B = len(feats) // beam
    tokens = np.zeros((B * beam, max_words + 2), dtype="int32"); tokens[:, 0] = TOKEN_START
    score = np.full((B, beam), -1e9); score[:, 0] = 0.0
    length = np.zeros((B, beam))
    done = np.zeros((B, beam), dtype=bool)
    for t in range(max_words + 1):
        logp = np.log(np.maximum(next_word_probs(feats, tokens[:, :t + 1]), 1e-12)).reshape(B, beam, V)
        logp[..., :2] = -1e9
        cand = np.where(done[..., None], -1e9, score[..., None] + logp)
        cand[..., 0] = np.where(done, score, -1e9)
        flat = cand.reshape(B, beam * V)
        best = np.argsort(-flat, axis=1)[:, :beam]
        src, word = best // V, best % V
        tokens = tokens[(np.arange(B)[:, None] * beam + src).ravel()]
        tokens[:, t + 1] = word.ravel()
        score = np.take_along_axis(flat, best, axis=1)
        was_done = np.take_along_axis(done, src, axis=1)
        length = np.take_along_axis(length, src, axis=1) + ~was_done
        done = was_done | (word == TOKEN_END)
        if done.all():
            break
    pick = (score / np.maximum(length, 1) ** alpha).argmax(axis=1)
    return [tokens_to_text(tokens[b * beam + pick[b]]) for b in range(B)]

def write_captions(features):
    fn = beam_captions if DECODING == "beam" else greedy_captions
    return sum((fn(features[s:s + 200]) for s in range(0, len(features), 200)), [])

# Attention maps (only for the attention model): where the network looks for each word
HAS_ATTENTION = any(layer.name == "decoder_attention" for layer in decoder.layers)
if HAS_ATTENTION:
    attention_model = keras.Model(decoder.inputs, decoder.get_layer("decoder_attention").output[-1])

def show_attention(image, caption, features, title="", cols=6):
    # Same as 03_captioning.ipynb (section 7.3): one small image per word, red = region used to write it
    words = caption.split()
    tokens = np.array([[TOKEN_START] + [VOCAB.index(w) for w in words]], dtype="int32")
    weights = np.asarray(attention_model({"transfer_values_input": np.asarray(features, dtype="float32")[None],
                                          "decoder_input": tokens}, training=False))[0, :len(words)]
    grid = int(np.sqrt(weights.shape[1]))   # 8
    h, w = image.shape[:2]
    rows = int(np.ceil((len(words) + 1) / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(3 * cols, 2.7 * rows), squeeze=False)
    axes = axes.ravel()
    axes[0].imshow(image); axes[0].set_title(title or "image", fontsize=11)
    for ax, word, wt in zip(axes[1:], words, weights):
        ax.imshow(image)
        ax.imshow(wt.reshape(grid, grid), cmap="jet", alpha=0.5, interpolation="bilinear", extent=(0, w, h, 0))
        ax.set_title(f"{word}  (max {wt.max():.2f})", fontsize=11)
    for ax in axes:
        ax.axis("off")
    plt.suptitle(caption, fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.show()

def show_attention_maps(table, photos, n=3):
    # Attention maps of the first n captioned photos (on the denoised image, the one the CNN saw)
    if not HAS_ATTENTION:
        print(f"  no attention maps: {CAPTION_MODEL} has no attention layer")
        return
    captions = table.set_index("path")["caption"]
    for j in range(min(n, len(photos["paths"]))):
        path = photos["paths"][j]
        show_attention(np.clip(photos["denoised"][j], 0, 1), captions[path], photos["features"][j], title=Path(path).name)

def list_images(folder):
    files = sorted(p for p in Path(folder).rglob("*") if p.suffix.lower() in IMAGE_EXTENSIONS)
    return [p.as_posix() for p in files if os.access(p, os.R_OK)]

def run_pipeline(paths, denoise_photos=DENOISE):
    # Full workflow on a list of files: one row per image
    paths = list(paths)
    timing = {}
    t0 = time.time()
    probs = classify(paths)
    timing["1. classification"] = time.time() - t0
    table = pd.DataFrame({"path": paths, "file_name": [Path(p).name for p in paths],
                          "class": [CLASS_NAMES[i] for i in probs.argmax(axis=1)],
                          "p_photo": probs[:, PHOTO].round(4), "caption": ""})
    is_photo = (probs.argmax(axis=1) == PHOTO)
    photo_paths = table.loc[is_photo, "path"].tolist()
    photos = load_01(photo_paths) if photo_paths else np.zeros((0,) + DENOISE_SIZE + (3,), "float32")
    if denoise_photos and len(photos):
        t0 = time.time()
        cleaned = denoise(photos)
        timing["2. denoising"] = time.time() - t0
    else:
        cleaned = photos
    if len(photos):
        t0 = time.time()
        feats = encode(cleaned)
        timing["3a. CNN encoder"] = time.time() - t0
        t0 = time.time()
        table.loc[is_photo, "caption"] = write_captions(feats)
        timing["3b. caption writing"] = time.time() - t0
    else:
        feats = None
    timing = pd.Series(timing, name="seconds").to_frame()
    timing["ms per image"] = timing["seconds"] / len(paths) * 1000
    return table, {"original": photos, "denoised": cleaned, "paths": photo_paths, "features": feats}, timing

print("[Block 3] OK")

## 4. Check on Images of Known Type

- **non-photos** from the test split of deliverable 1 (not seen in training): they must be set aside;
- **photos** from COCO val2017 and from `data_raw/Dataset` (noisy scans): they must get a caption.

In [ ]:
print("[Block 4] Self-check : starting...")

from sklearn.model_selection import train_test_split

def deliverable1_test_split():
    # Test images of deliverable 1 (same cleaning and split as 01_photo_ou_pas.ipynb)
    index = pd.read_csv(DATA / "index.csv")
    index = index[index["category"].isin(CLASS_NAMES)]
    index = index[index["readable"] & (index[["width", "height"]].min(axis=1) >= 32)
                  & ~index.duplicated("md5", keep="first")]
    _, rest = train_test_split(index, test_size=0.30, stratify=index["category"], random_state=42)
    _, test = train_test_split(rest, test_size=0.50, stratify=rest["category"], random_state=42)
    return test

rng = np.random.default_rng(42)
check = []
if (DATA / "index.csv").exists():
    test_d1 = deliverable1_test_split()
    for category, part in test_d1[test_d1["category"] != "Photo"].groupby("category"):
        readable = [p for p in part["path"] if os.access(p, os.R_OK)]
        check += [(p, category) for p in rng.choice(readable, size=min(20, len(readable)), replace=False)]
coco_val = list_images(RAW / "coco" / "val2017")
check += [(p, "Photo (COCO)") for p in rng.choice(coco_val, size=min(30, len(coco_val)), replace=False)]
noisy = list_images(RAW / "Dataset")
check += [(p, "Photo (noisy scan)") for p in rng.choice(noisy, size=min(30, len(noisy)), replace=False)]

check_table, _, _ = run_pipeline([p for p, _ in check])
check_table["true type"] = [t for _, t in check]
check_table["captioned"] = check_table["caption"] != ""
summary = check_table.groupby("true type").agg(images=("path", "size"), sent_to_captioning=("captioned", "mean"))
display(summary.round(3))
print("  not a photo, but captioned:", check_table[~check_table["true type"].str.startswith("Photo") & check_table.captioned]["file_name"].tolist())
print("  photo, but set aside:", check_table[check_table["true type"].str.startswith("Photo") & ~check_table.captioned][["file_name", "class"]].values.tolist())

print("[Block 4] OK")

## 5. Running the Workflow on `INPUT_DIR`

In [ ]:
print("[Block 5] Workflow : starting...")

def process_folder(folder):
    paths = list_images(folder)
    print(f"  {folder}: {len(paths)} images")
    table, photos, timing = run_pipeline(paths)
    out = RESULTS / f"pipeline_{Path(folder).name}.csv"
    table.drop(columns="path").to_csv(out, index=False)
    print(f"  {table['caption'].ne('').sum()} photos captioned, {table['caption'].eq('').sum()} images set aside "
          f"-> {out}")
    display(table["class"].value_counts().rename("images").to_frame())
    display(timing.round(3))
    print(f"  total: {timing['ms per image'].sum():.1f} ms per image")
    return table, photos

def show_gallery(table, photos, n=8):
    # Photos (input / denoised + caption), then images set aside
    k = min(n, len(photos["paths"]))
    if k:
        fig, axes = plt.subplots(2, k, figsize=(3.6 * k, 8), squeeze=False)
        captions = table.set_index("path").loc[photos["paths"][:k], "caption"]
        for j in range(k):
            axes[0, j].imshow(photos["original"][j]); axes[0, j].axis("off"); axes[0, j].set_title("input", fontsize=9)
            axes[1, j].imshow(np.clip(photos["denoised"][j], 0, 1)); axes[1, j].axis("off")
            cap = captions.iloc[j]
            axes[1, j].set_title("\n".join(cap[i:i + 30] for i in range(0, len(cap), 30)), fontsize=9)
        plt.suptitle("Photos: input (top) -> denoised + caption (bottom)", fontweight="bold")
        plt.tight_layout(); plt.show()
    aside = table[table["caption"] == ""].head(n)
    if len(aside):
        fig, axes = plt.subplots(1, len(aside), figsize=(3 * len(aside), 3.4), squeeze=False)
        for ax, (_, row) in zip(axes[0], aside.iterrows()):
            ax.imshow(iio.imread(row["path"]), cmap="gray"); ax.axis("off")
            ax.set_title(f"{row['class']}\np(photo) = {row['p_photo']:.2f}", fontsize=9)
        plt.suptitle("Set aside (not a photo)", fontweight="bold")
        plt.tight_layout(); plt.show()

table, photos = process_folder(INPUT_DIR)
display(table[["file_name", "class", "p_photo", "caption"]].head(15))
show_gallery(table, photos)
show_attention_maps(table, photos)

print("[Block 5] OK")

## 6. Score Against Human Captions (if available)

`ANNOTATIONS`: a COCO `.json` file, or a `.csv` with columns `file_name`, `caption`. Score: BLEU-1 to 4.

In [ ]:
print("[Block 6] Scoring : starting...")

def clean_caption(text):
    return " ".join(re.sub(r"[^a-z0-9 ]", " ", str(text).lower()).split())

def load_references(path):
    # file name -> list of human captions
    path = Path(path)
    if path.suffix == ".json":
        with open(path) as f:
            ann = json.load(f)
        names = {im["id"]: im["file_name"] for im in ann["images"]}
        refs = pd.DataFrame([(names[a["image_id"]], a["caption"]) for a in ann["annotations"]],
                            columns=["file_name", "caption"])
    else:
        refs = pd.read_csv(path)
    return refs.groupby("file_name")["caption"].apply(lambda c: [clean_caption(x) for x in c]).to_dict()

def score_captions(table, annotations):
    refs = load_references(annotations)
    scored = table[(table["caption"] != "") & table["file_name"].isin(refs)]
    print(f"  {len(scored)} captioned images with human captions "
          f"({table['file_name'].isin(refs).sum()} images of the folder have human captions)")
    if not len(scored):
        return None
    r = [[c.split() for c in refs[f]] for f in scored["file_name"]]
    h = [c.split() for c in scored["caption"]]
    bleu = {f"BLEU-{n}": corpus_bleu(r, h, weights=tuple([1 / n] * n)) for n in (1, 2, 3, 4)}
    display(pd.DataFrame([bleu], index=[Path(annotations).name]).round(4))
    return bleu

if ANNOTATIONS is not None:
    score_captions(table, ANNOTATIONS)
else:
    print("  ANNOTATIONS = None: no human captions to compare with")

print("[Block 6] OK")

## 7. Production and Maintenance

- **Environment**: `make up` starts the Docker container (TensorFlow 2.21 + GPU, Jupyter on port 8888).
- **Needed files**: the 3 `.keras` models and `caption_<model>_vocab.json` in `results/` (InceptionV3 is downloaded by Keras).
- **Retraining**: each model is retrained in its own notebook; keep the same file names. The vocabulary and the
  captioning model must come from the same run (checked in block 2).
- **New data**: add TouNum images with captions in COCO format and retrain `03_captioning.ipynb`.
- **Speed**: see the timing table of section 5.

## 8. Demo Day

Change the folder (and the captions file if any), then run this cell.

In [ ]:
print("[Block 8] Demo : starting...")

INPUT_DIR = RAW / "Dataset"   # <- the folder given on demo day
ANNOTATIONS = None            # <- its captions file, if any (COCO .json or .csv file_name, caption)

demo_table, demo_photos = process_folder(INPUT_DIR)
display(demo_table[["file_name", "class", "p_photo", "caption"]])
show_gallery(demo_table, demo_photos, n=8)
show_attention_maps(demo_table, demo_photos, n=3)   # where the network looks (attention model only)
if ANNOTATIONS is not None:
    score_captions(demo_table, ANNOTATIONS)

print("[Block 8] OK")